## Notebook 12 — Gaussian Noise Feature Analysis
**Project:** Machine Learning for High Performance Optical Sorting
**Author:** Mohamed Tawfeek
**Description:** Analysis of how Gaussian noise perturbs the HSV+LBP feature space — per-feature shifts and histogram entropy across two severity levels


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
import cv2
from PIL import Image
from skimage.feature import local_binary_pattern
from sklearn.model_selection import train_test_split

import sys
sys.path.insert(0, os.path.abspath('..'))

In [2]:
IMAGE_SIZE     = (224, 224)
HISTOGRAM_BINS = 32
CLASSES        = ['glass', 'paper', 'cardboard', 'plastic', 'metal', 'trash']
DATASET_PATH   = os.path.join("..", "datasets", "dataset-resized")
FIGURES_DIR    = os.path.join("..", "results", "figures")

In [3]:
# Gaussian noise perturbation function and the feature extraction pipeline used to re-extract features under each noise condition

from src.perturbations import apply_gaussian_noise, make_perturbation
from src.features import extract_features_from_image as extract_features
from src.features import load_dataset


def load_dataset_sorted(perturb_fn=None):
    return load_dataset(DATASET_PATH, perturb_fn=perturb_fn, classes=CLASSES)


In [4]:
# Loads the full clean dataset and recreates the standard split to isolate the 380-image test partition as the reference condition

print("Loading clean features (sorted order)...")
X_all, y_all = load_dataset_sorted(perturb_fn=None)
 
X_train_val, X_test_clean, y_train_val, y_test = train_test_split(
    X_all, y_all, test_size=0.15, random_state=42, stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
)
print(f"Test set: {X_test_clean.shape[0]} images")

Loading clean features (sorted order)...


Test set: 380 images


In [5]:
# Re-extracts features under mild (S1, σ=5) and severe (S5, σ=75) Gaussian noise, retaining only the test partition for analysis

print("Loading S1 noisy features (sigma=5)...")
X_all_s1, _ = load_dataset_sorted(
    perturb_fn=make_perturbation('gaussian_noise', 1)
)
_, X_test_s1, _, _ = train_test_split(
    X_all_s1, y_all, test_size=0.15, random_state=42, stratify=y_all
)
 
print("Loading S5 noisy features (sigma=75)...")
X_all_s5, _ = load_dataset_sorted(
    perturb_fn=make_perturbation('gaussian_noise', 5)
)
_, X_test_s5, _, _ = train_test_split(
    X_all_s5, y_all, test_size=0.15, random_state=42, stratify=y_all
)
 
print(f"All three conditions loaded. Test set shape: {X_test_clean.shape}")

Loading S1 noisy features (sigma=5)...


Loading S5 noisy features (sigma=75)...


All three conditions loaded. Test set shape: (380, 122)


In [6]:
groups = {"H": (0, 32), "S": (32, 64), "V": (64, 96), "LBP": (96, 122)}
group_colors = {"H": "#2196F3", "S": "#4CAF50", "V": "#FF9800", "LBP": "#9C27B0"}

In [7]:
# Computes mean absolute per-feature shift between clean and noisy test sets, summarised per channel group

print("\nComputing feature-level shifts...")
 
def group_stats(X_clean, X_noisy, groups):
    diff = np.abs(X_noisy - X_clean)   # (n_test, 122)
    stats = {}
    for name, (lo, hi) in groups.items():
        block = diff[:, lo:hi]
        stats[name] = {
            "mean_shift":   block.mean(),
            "median_shift": np.median(block),
            "max_shift":    block.max(),
            "per_feature":  block.mean(axis=0),   # mean shift per bin
        }
    return stats
 
stats_s1 = group_stats(X_test_clean, X_test_s1, groups)
stats_s5 = group_stats(X_test_clean, X_test_s5, groups)
 
print("\n" + "=" * 65)
print("MEAN ABSOLUTE FEATURE SHIFT PER GROUP")
print("=" * 65)
print(f"{'Group':<8} {'S1 (σ=5)':>12} {'S5 (σ=75)':>12} {'Ratio S5/S1':>12}")
print("-" * 65)
for name in groups:
    s1 = stats_s1[name]["mean_shift"]
    s5 = stats_s5[name]["mean_shift"]
    print(f"{name:<8} {s1:>12.5f} {s5:>12.5f} {s5/s1:>12.1f}x")


Computing feature-level shifts...

MEAN ABSOLUTE FEATURE SHIFT PER GROUP
Group        S1 (σ=5)    S5 (σ=75)  Ratio S5/S1
-----------------------------------------------------------------
H             0.02584      0.12317          4.8x
S             0.01180      0.11817         10.0x
V             0.00521      0.05499         10.6x
LBP           0.02287      0.03586          1.6x


In [8]:
# Measures mean Shannon entropy of histogram bins per group as an indicator of how much noise flattens the feature distributions

print("\nComputing histogram flatness (entropy-based)...")
 
def histogram_entropy(X, lo, hi):
    block = X[:, lo:hi]
    block = block / (block.sum(axis=1, keepdims=True) + 1e-10)
    with np.errstate(divide="ignore", invalid="ignore"):
        log_block = np.where(block > 0, np.log(block), 0)
    entropy = -(block * log_block).sum(axis=1)
    return entropy.mean()
 
print(f"\n{'Group':<8} {'Clean entropy':>15} {'S1 entropy':>12} {'S5 entropy':>12}")
print("-" * 52)
for name, (lo, hi) in groups.items():
    e_clean = histogram_entropy(X_test_clean, lo, hi)
    e_s1    = histogram_entropy(X_test_s1,    lo, hi)
    e_s5    = histogram_entropy(X_test_s5,    lo, hi)
    print(f"{name:<8} {e_clean:>15.4f} {e_s1:>12.4f} {e_s5:>12.4f}")


Computing histogram flatness (entropy-based)...

Group      Clean entropy   S1 entropy   S5 entropy
----------------------------------------------------
H                 1.3233       1.7582       3.0008
S                 1.7813       1.9321       2.9585
V                 2.6596       2.6813       2.8597
LBP               2.6351       2.0349       1.4713


In [9]:
# Four-panel figure comparing mean feature vectors under clean, mild, and severe noise conditions across all channel groups

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
 
for ax, (name, (lo, hi)) in zip(axes, groups.items()):
    x = np.arange(hi - lo)
    clean_mean = X_test_clean[:, lo:hi].mean(axis=0)
    s1_mean    = X_test_s1[:,    lo:hi].mean(axis=0)
    s5_mean    = X_test_s5[:,    lo:hi].mean(axis=0)
 
    ax.plot(x, clean_mean, color=group_colors[name],
            linewidth=2,   label="Clean",        zorder=3)
    ax.plot(x, s1_mean,    color="orange",
            linewidth=1.5, label="S1 (σ=5)",    linestyle="--", zorder=2)
    ax.plot(x, s5_mean,    color="red",
            linewidth=1.5, label="S5 (σ=75)",   linestyle=":",  zorder=1)
 
    ax.set_title(f"{name} Channel — Mean Feature Vector", fontsize=10,
                 fontweight="bold", color=group_colors[name])
    ax.set_xlabel("Bin index", fontsize=8)
    ax.set_ylabel("Mean normalised value", fontsize=8)
    ax.legend(fontsize=7.5)
    ax.set_ylim(bottom=0)
 
plt.suptitle(
    "Effect of Gaussian Noise on Mean Feature Vectors (test set, n=380)\n"
    "Clean vs S1 (σ=5) vs S5 (σ=75)",
    fontsize=11
)
plt.tight_layout()
out1 = os.path.join(FIGURES_DIR, "gaussian_noise_feature_shift.png")
plt.savefig(out1, dpi=150, bbox_inches="tight")
plt.close()
print(f"\nSaved: {out1}")


Saved: ..\results\figures\gaussian_noise_feature_shift.png


In [10]:
# Stacked bar chart of per-feature mean absolute shift for S1 and S5, coloured by channel group

fig, ax = plt.subplots(figsize=(14, 4))
 
shift_s1 = np.abs(X_test_s1    - X_test_clean).mean(axis=0)
shift_s5 = np.abs(X_test_s5    - X_test_clean).mean(axis=0)
 
bar_colors = (["#2196F3"] * 32 + ["#4CAF50"] * 32 +
              ["#FF9800"] * 32 + ["#9C27B0"] * 26)
 
ax.bar(np.arange(122), shift_s5, color=bar_colors,
       alpha=0.5, width=1.0, linewidth=0, label="S5 (σ=75)")
ax.bar(np.arange(122), shift_s1, color=bar_colors,
       alpha=1.0, width=1.0, linewidth=0, label="S1 (σ=5)")
 
for b, label in [(32, "H|S"), (64, "S|V"), (96, "V|LBP")]:
    ax.axvline(b - 0.5, color="black", linewidth=1.2,
               linestyle="--", alpha=0.5)
 
ax.set_xlabel("Feature Index", fontsize=10)
ax.set_ylabel("Mean |Δ feature value|", fontsize=10)
ax.set_title(
    "Per-Feature Mean Absolute Shift Under Gaussian Noise\n"
    "Coloured by group: H (blue), S (green), V (orange), LBP (purple)",
    fontsize=10
)
ax.legend(fontsize=9)
 
for name, (lo, hi) in groups.items():
    ax.text((lo + hi) / 2, ax.get_ylim()[1] * 0.92,
            name, ha="center", fontsize=9,
            fontweight="bold", color=group_colors[name])
 
plt.tight_layout()
out2 = os.path.join(FIGURES_DIR, "gaussian_noise_per_feature_shift.png")
plt.savefig(out2, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out2}")


Saved: ..\results\figures\gaussian_noise_per_feature_shift.png


In [11]:
# Grouped bar chart comparing mean absolute feature shift per channel group between the two noise severities

fig, ax = plt.subplots(figsize=(7, 4))
x      = np.arange(len(groups))
width  = 0.35
names  = list(groups.keys())
s1_vals = [stats_s1[n]["mean_shift"] for n in names]
s5_vals = [stats_s5[n]["mean_shift"] for n in names]
 
bars1 = ax.bar(x - width/2, s1_vals, width, label="S1 (σ=5)",
               color=[group_colors[n] for n in names], alpha=1.0)
bars2 = ax.bar(x + width/2, s5_vals, width, label="S5 (σ=75)",
               color=[group_colors[n] for n in names], alpha=0.45)
 
for bar, val in zip(bars1, s1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
            f"{val:.4f}", ha="center", va="bottom", fontsize=7.5)
for bar, val in zip(bars2, s5_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
            f"{val:.4f}", ha="center", va="bottom", fontsize=7.5)
 
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel("Mean |Δ feature value|", fontsize=10)
ax.set_title("Group-Level Feature Shift Under Gaussian Noise", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
out3 = os.path.join(FIGURES_DIR, "gaussian_noise_group_shift.png")
plt.savefig(out3, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {out3}")

Saved: ..\results\figures\gaussian_noise_group_shift.png


In [12]:
print("\n" + "=" * 30)
print("KEY FINDINGS")
print("=" * 30)
print("\nGroup-level mean absolute shift:")
for name in groups:
    s1 = stats_s1[name]["mean_shift"]
    s5 = stats_s5[name]["mean_shift"]
    print(f"  {name:<6}: S1={s1:.5f}  S5={s5:.5f}  ratio={s5/s1:.1f}x")
 
print("\nHistogram entropy (higher = flatter = more uniform = signal destroyed):")
for name, (lo, hi) in groups.items():
    e_clean = histogram_entropy(X_test_clean, lo, hi)
    e_s1    = histogram_entropy(X_test_s1,    lo, hi)
    e_s5    = histogram_entropy(X_test_s5,    lo, hi)
    print(f"  {name:<6}: clean={e_clean:.4f}  S1={e_s1:.4f}  S5={e_s5:.4f}  "
          f"  S1 increase={e_s1-e_clean:+.4f}  S5 increase={e_s5-e_clean:+.4f}")


KEY FINDINGS

Group-level mean absolute shift:
  H     : S1=0.02584  S5=0.12317  ratio=4.8x
  S     : S1=0.01180  S5=0.11817  ratio=10.0x
  V     : S1=0.00521  S5=0.05499  ratio=10.6x
  LBP   : S1=0.02287  S5=0.03586  ratio=1.6x

Histogram entropy (higher = flatter = more uniform = signal destroyed):
  H     : clean=1.3233  S1=1.7582  S5=3.0008    S1 increase=+0.4349  S5 increase=+1.6775
  S     : clean=1.7813  S1=1.9321  S5=2.9585    S1 increase=+0.1508  S5 increase=+1.1772
  V     : clean=2.6596  S1=2.6813  S5=2.8597    S1 increase=+0.0217  S5 increase=+0.2001
  LBP   : clean=2.6351  S1=2.0349  S5=1.4713    S1 increase=-0.6002  S5 increase=-1.1638
